In [176]:
from dotenv import load_dotenv
import openai
import pandas as pd

load_dotenv()
openai_client = openai.OpenAI()

### Load the knowledge base

In [177]:
import sys
sys.path.append('../src')

from load_index import load_faq_data

documentation = load_faq_data()


In [ ]:
print(documentation)

In [179]:
df = pd.DataFrame(documentation)
df.head()


,id,manual,title,text,url
0,ctm60060,company-taxation-manual,Close companies: general: broad definition,"Subject to certain exceptions, a close company...",https://www.gov.uk/hmrc-internal-manuals/compa...
1,ctm03570,company-taxation-manual,Corporation Tax: small profits relief: associa...,A company is an 'associated company' of anothe...,https://www.gov.uk/hmrc-internal-manuals/compa...
2,ctm61505,company-taxation-manual,Close companies: loans to participators and ar...,CTA10 Part 10 CTA10/S455 applies to loans/adva...,https://www.gov.uk/hmrc-internal-manuals/compa...
3,ctm15205,company-taxation-manual,"Distributions: general: dividends, distributio...",Companies Act 2006 Introduction References are...,https://www.gov.uk/hmrc-internal-manuals/compa...
4,ctm03580,company-taxation-manual,Corporation Tax: small profits relief: associa...,CTM03710 onwards gives detailed guidance on de...,https://www.gov.uk/hmrc-internal-manuals/compa...


In [180]:
df.columns.tolist()

['id', 'manual', 'title', 'text', 'url']

In [181]:
df.manual.value_counts()

manual
paye-manual                  279
vat-registration-manual      263
company-taxation-manual      259
capital-gains-manual         253
capital-allowances-manual    245
Name: count, dtype: int64

In [182]:
len(documentation)

1299

In [183]:
documentation[100]

{'id': 'ctm61555',
 'manual': 'company-taxation-manual',
 'title': 'Close companies: loans to participators: indirect loans: total income',
 'text': 'CTA10/S459 (3) (b) A payment, transfer, release etc, arising from an arrangement described in CTM61550 may be included in the total income of the recipient of the payment. When the amount included in that person’s total income is more than or equal to the amount of the loan or advance, CTA10/S459 will not apply. This may happen, for example, where: a payment to the recipient is treated as his/her income for IT purposes (for example, as a distribution within CTA10/S1000 (1) B or CTA10/S1064, or a loan to the recipient falls within the provisions of ITTOIA05/S633 The ‘total income’ of an individual is defined in ITA07/S23 as the sum of the amounts of income on which the taxpayer is charged to income tax for the tax year. Broadly speaking it consists of earned and investment (or unearned) income. It does not include anything on which capital

In [184]:
documentation[100]['id']

'ctm61555'

In [185]:
documentation[100]['manual']

'company-taxation-manual'

In [186]:
documentation[100]['title']

'Close companies: loans to participators: indirect loans: total income'

### index the knowledge base

In [187]:
import minsearch

define the text field for the index for optimal search results. The text field should be a combination of the title and text fields. The keyword field should be the manual field.

In [188]:
from minsearch import Index

index = Index(
    text_fields=["title", "text"],
    keyword_fields=["manual"])

index.fit(documentation)

In [189]:
question = "What counts as capital expenditure for capital allowances purposes?"

In [190]:
df.head()

,id,manual,title,text,url
0,ctm60060,company-taxation-manual,Close companies: general: broad definition,"Subject to certain exceptions, a close company...",https://www.gov.uk/hmrc-internal-manuals/compa...
1,ctm03570,company-taxation-manual,Corporation Tax: small profits relief: associa...,A company is an 'associated company' of anothe...,https://www.gov.uk/hmrc-internal-manuals/compa...
2,ctm61505,company-taxation-manual,Close companies: loans to participators and ar...,CTA10 Part 10 CTA10/S455 applies to loans/adva...,https://www.gov.uk/hmrc-internal-manuals/compa...
3,ctm15205,company-taxation-manual,"Distributions: general: dividends, distributio...",Companies Act 2006 Introduction References are...,https://www.gov.uk/hmrc-internal-manuals/compa...
4,ctm03580,company-taxation-manual,Corporation Tax: small profits relief: associa...,CTM03710 onwards gives detailed guidance on de...,https://www.gov.uk/hmrc-internal-manuals/compa...


Running the filter search using the idex 

In [191]:
search_results = index.search(
    question, 
    boost_dict={"title": 1.0, "text": 2.0},
    filter_dict={"manual": "company-taxation-manual"}, 
    num_results=5
    )

search_results

[{'id': 'ctm00512',
  'manual': 'company-taxation-manual',
  'title': 'Introductory: meaning of ordinary share capital: share capital',
  'text': 'The UK Companies Acts do not define share capital directly but state that references in the Companies Acts to a company having a share capital are to a power under its constitution to issue shares.\xa0 Share capital is the amount appearing in the balance sheet from the issue of shares and if shares are issued at a premium over their nominal value the premium is reflected in a share premium account separate from the nominal share capital, although for both company law purposes (CA06/S610 (4)) and tax law purposes (CTA10/S1025 (2)) the premium is treated as forming part of the share capital for the purposes of its “reduction” under company law and its “repayment” under tax law. This means that share capital reflects the equity (ownership) interests of the company members in their company and, in liquidation, the value of the interests (if any)

In [192]:
search_results = index.search(
    question, 
    boost_dict={"title": 1.0, "text": 2.0},
    filter_dict={"manual": "vat-registration-manual"}, 
    num_results=5
    )

search_results

[{'id': 'vatreg02650',
  'manual': 'vat-registration-manual',
  'title': 'Basic principles of registration: definition of a capital asset',
  'text': 'The case of Verbond van Nederlandse Ondernemingen v Inspecteur der Invoerrechten en Accijnzen CJEC [1977] 1 CMLR 413, heard by the CJEC gave a broad definition of a capital good: it may be tangible or intangible it must be used for the purpose of some business activity it must be durable it must be of such value that its acquisition costs are not treated as current expenditure but are written off over a number of years. The term ‘capital assets’ therefore covers goods and services which have been used in a business as ‘equipment’. It includes premises, plant, machinery, office machinery, computers, office furniture, used company cars, patent rights which have been exploited and which are sold outright, and goodwill. Legal cases on capital assets The Tribunal in the case of The Trustees of the Mellerstain Trust, EDN/89/41 VTD 4256 decided

Defining a function to search the index with a question and a course name. The function should return the top 5 results from the index.

In [193]:
def main_search(question, manual="capital-allowances-manual"):
    boost_dict = {"text": 2.0, "title": 1.0}
    filter_dict = {"manual": manual}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
    
    
search_results = main_search(question)
search_results

[{'id': 'ca11530',
  'manual': 'capital-allowances-manual',
  'title': 'General: Definitions: Capital expenditure and capital sums',
  'text': 'CAA01/S4 & CAA01/S572 (3)(b) An amount that the payer can treat as a deduction in computing business profits or which can be allowed as a deduction from the taxable earnings of an employment or office held by the payer is not capital expenditure. An amount that is treated by the recipient as a business receipt or employment income is not a capital sum. Annual payments from which tax should be deducted under ITA/Chapter 6 Part 15 or s906 are not capital expenditure or capital sums. If property is exchanged or if a leasehold interest is surrendered for valuable consideration the transaction is treated as a sale. Where the consideration is not in money, the part of it that would have been a capital sum if it had been paid in money is treated as a capital sum. The purchase price of an asset sometimes includes VAT. If the VAT is allowable as input t

Another use of the search function to search for a question in a different course. The function should return the top 5 results from the index. In this case, using the capital gain manual

In [194]:
def main_search(question, manual="company-taxation-manual"):
    boost_dict = {"text": 2.0, "title": 1.0}
    filter_dict = {"manual": manual}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
    
search_results = main_search(question)
search_results


[{'id': 'ctm00512',
  'manual': 'company-taxation-manual',
  'title': 'Introductory: meaning of ordinary share capital: share capital',
  'text': 'The UK Companies Acts do not define share capital directly but state that references in the Companies Acts to a company having a share capital are to a power under its constitution to issue shares.\xa0 Share capital is the amount appearing in the balance sheet from the issue of shares and if shares are issued at a premium over their nominal value the premium is reflected in a share premium account separate from the nominal share capital, although for both company law purposes (CA06/S610 (4)) and tax law purposes (CTA10/S1025 (2)) the premium is treated as forming part of the share capital for the purposes of its “reduction” under company law and its “repayment” under tax law. This means that share capital reflects the equity (ownership) interests of the company members in their company and, in liquidation, the value of the interests (if any)

----

### Buikding the LLM prompt 

In [195]:
INSTRUCTIONS = """
Your task is to answer questions from the manual documents
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with \"I don't know.\"
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [196]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["title"])
        lines.append("guide: " + doc["text"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()



In [197]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
What counts as capital expenditure for capital allowances purposes?

Context:
Introductory: meaning of ordinary share capital: share capital
guide: The UK Companies Acts do not define share capital directly but state that references in the Companies Acts to a company having a share capital are to a power under its constitution to issue shares.  Share capital is the amount appearing in the balance sheet from the issue of shares and if shares are issued at a premium over their nominal value the premium is reflected in a share premium account separate from the nominal share capital, although for both company law purposes (CA06/S610 (4)) and tax law purposes (CTA10/S1025 (2)) the premium is treated as forming part of the share capital for the purposes of its “reduction” under company law and its “repayment” under tax law. This means that share capital reflects the equity (ownership) interests of the company members in their company and, in liquidation, the value of the interests 

----

In [198]:
# We already built `prompt` in the previous section.
## prompt = build_prompt(question, search_results)

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

# The easiest way to read the model answer
print(response.output_text)
print("")

# Optional: inspect token usage
print(response.usage)
print("")

# Example cost estimate for gpt-5.4-mini
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

print(f"Estimated cost: ${cost:.8f}")

For capital allowances purposes, **capital expenditure** is generally expenditure that is **capital in nature rather than revenue in nature**.

In practical terms, it is usually expenditure on:

- **acquiring, constructing or improving** an asset used in the business, such as:
  - plant and machinery
  - buildings or structures that qualify
  - fixtures
  - integral features
  - vehicles and equipment
- **bringing an asset into existence**
- **enhancing or extending** the life, value, or capacity of an existing capital asset

It is **not** usually capital expenditure if it is ordinary day-to-day business spending, such as:

- repairs and maintenance
- staff costs
- rent, rates, and utilities
- routine replacement of consumables
- other revenue expenses

A helpful rule of thumb is:

- **capital expenditure** = spending on a **lasting asset or benefit**
- **revenue expenditure** = spending on the **running of the business**

For capital allowances, the expenditure must usually be incurre

next, let us combined all the code and build a modular function that can be used to search the index and return the top 5 results. The function should take in a question and a course name as parameters. The function should return the top 5 results from the index.

In [210]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history,
    )

    return response.output_text


def rag(query, manual='paye-manual', model="gpt-5.4-mini"):
    search_results = main_search(query, manual=manual)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


In [211]:
answer = rag("How do you gross up a payment for PAYE purposes?")
print(answer)

To gross up a payment for PAYE purposes, calculate the amount needed so the employee ends up with the same net amount after tax and primary Class 1 NICs are deducted.

The method is:

1. Work out the tax and primary Class 1 NICs due on the original amount.
2. Add the employee’s tax rate and primary Class 1 NICs rate together.
3. Subtract that total from 100.
4. Multiply the deductions by 100, then divide by the result from step 3.
5. Add that grossed-up amount to the original expense/benefit.
6. Pay that total through payroll and deduct PAYE and NICs from it.

Example from the guide:
- Original amount: £567
- Tax at 20% = £113.40
- Primary Class 1 NICs at 8% = £45.36
- Total deductions = £158.76

Gross-up calculation:
- £158.76 x 100 = £15,876
- 20 + 8 = 28
- 100 - 28 = 72
- £15,876 / 72 = £220.50

So the grossed-up payment is £220.50, which is added to the original £567, giving £787.50 to be paid through payroll.


In [206]:
df.manual.value_counts().to_dict()

{'paye-manual': 279,
 'vat-registration-manual': 263,
 'company-taxation-manual': 259,
 'capital-gains-manual': 253,
 'capital-allowances-manual': 245}

In [209]:
answer2 = rag( "What are the rules for estimated pay when coding an employee?")
print(answer2)

Estimated pay rules depend on the employment/pension type and whether RTI information is available.

### Main rules
- **Estimated pay is required** for every employment and occupational pension.
- The system usually uses the **latest FPS (RTI)** to calculate it.
- If the latest FPS cannot be used, it will use:
  1. data already held on the system, or
  2. default values.

### If RTI is used, the calculation depends on pay frequency

#### Regular pay
Regular means not marked **Paid Irregularly** or **Paid Annually**.

- **Continuous employment/pension started before CY**
  - Weekly / fortnightly / 4-weekly:  
    taxable pay to date ÷ week number × 52
  - Monthly:  
    taxable pay to date ÷ month number × 12

- **Mid-year start with a start date**
  - Weekly: pay in period × (52 - week number) + pay in period
  - Fortnightly: pay in period × ((52 - week number) ÷ 2) + pay in period
  - 4-weekly: pay in period × ((52 - week number) ÷ 4) + pay in period
  - If week number is **52 or more